# Agentic ecommerce search with BM25 + E5 (min tool calls)

We build two search tools (BM25 and E5 embeddings), add a guard that discourages repeated or very similar queries, and wrap everything in an agentic loop.

ELI5: the agent is a student who must use at least 6 search attempts before it can stop. We also tell it not to ask nearly the same question twice.

In [1]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@ee2526eb8bfac087dc3f90522cc7191032e47dfd
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

  Cloning https://github.com/softwaredoug/cheat-at-search.git (to revision ee2526eb8bfac087dc3f90522cc7191032e47dfd) to /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-p_qgy_xg
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-p_qgy_xg
  Running command git rev-parse -q --verify 'sha^ee2526eb8bfac087dc3f90522cc7191032e47dfd'
  Running command git fetch -q https://github.com/softwaredoug/cheat-at-search.git ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
2026-05-25 21:17:48,716 - cheat_at_s

## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [2]:
import logging
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("agentic_ecom")

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

corpus[["doc_id", "title", "description"]].head(3)

,doc_id,title,description
0,0,solid wood platform bed,"good , deep sleep can be quite difficult to ha..."
1,1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend..."
2,2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...


## Sample queries (8-16)

We use a small slice of queries to keep the notebook fast and easy to run.

In [3]:
QUERY_COUNT = 12
queries = judgments[["query", "query_id"]].drop_duplicates()
queries = queries.sample(n=QUERY_COUNT, random_state=7).reset_index(drop=True)
queries

,query,query_id
0,body pillow case,118
1,deer coat hooks,455
2,tye dye duvet cover,459
3,dining table vinyl cloth,384
4,ligth bulb,305
5,garage sports storage rack,469
6,zodiac pillow,120
7,podium with locking cabinet,141
8,shoe closet,435
9,outdoor privacy wall,13


## Tool 1: BM25 search (ELI5)

BM25 is a classic keyword search. We build a text index on titles and descriptions, then score documents for a query.

ELI5: BM25 is like counting important words in each product and ranking the ones that match your query best.

In [4]:
from searcharray import SearchArray
from searcharray.similarity import bm25_similarity
from cheat_at_search.tokenizers import snowball_tokenizer

BM25_K1 = 1.2
BM25_B = 0.75
TITLE_BOOST = 10.0
DESCRIPTION_BOOST = 1.0

title_index = SearchArray.index(
    corpus["title"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)
description_index = SearchArray.index(
    corpus["description"].fillna(""),
    tokenizer=snowball_tokenizer,
    workers=4,
)

def bm25_search_tool(query, top_k=10, agent_state=None):
    tokens = snowball_tokenizer(query)
    if not tokens:
        return []
    similarity = bm25_similarity(k1=BM25_K1, b=BM25_B)
    title_scores = title_index.score(tokens, similarity=similarity)
    description_scores = description_index.score(tokens, similarity=similarity)
    scores = TITLE_BOOST * title_scores + DESCRIPTION_BOOST * description_scores
    top_k = min(top_k, len(scores))
    if top_k == 0:
        return []
    top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
    top_sorted = top_idx[np.argsort(-scores[top_idx])]
    results = []
    for idx in top_sorted:
        row = corpus.iloc[idx]
        results.append({
            "id": row.get("doc_id", idx),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": float(scores[idx]),
        })
    return results

bm25_search_tool("salon chair", top_k=3)

2026-05-25 21:17:53,463 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-05-25 21:17:53,465 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-05-25 21:17:53,466 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

2026-05-25 21:17:56,169 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-05-25 21:17:56,230 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


[{'id': np.int64(7465),
  'title': 'hair salon chair',
  'description': 'offers a wide selection of professional salon products including styling chair , salon hairdryer , salon equipment , etc . to showcase the uniqueness of each salon',
  'score': 52.17097473144531},
 {'id': np.int64(22130),
  'title': 'height-adjustable stool salon chair',
  'description': 'here comes the stool , with 360-degree chair wheels , by which you can move freely with no noise in your space . with the comfortable table seat design , you will get a comfortable sitting experience . due to the stable and durable construction — the explosive gas rod passed germany tuv , and the high-strength steel frame — guarantees safer and longer use . top back to get one for your salon , dining table , bar table , etc .',
  'score': 44.94427490234375},
 {'id': np.int64(33689),
  'title': 'office chair work beauty salon chair brown',
  'description': '',
  'score': 39.476078033447266}]

## Tool 2: E5 embeddings search (ELI5)

Embeddings turn text into vectors. We can compare vectors to find semantically similar products.

ELI5: embeddings turn sentences into numbers, and we find items with numbers that look similar.

In [5]:
!pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "intfloat/e5-base-v2"
EMBEDDING_BATCH = 64
EMBEDDING_MAX_DOCS = None  # set to e.g. 5000 for faster iteration

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

def _row_text(row):
    title = row.get("title")
    description = row.get("description")
    title_text = title.strip() if isinstance(title, str) else ""
    description_text = description.strip() if isinstance(description, str) else ""
    if title_text and description_text:
        return f"{title_text}\n\n{description_text}"
    if title_text:
        return title_text
    return description_text

def embed_texts(texts):
    embeddings = []
    for start in range(0, len(texts), EMBEDDING_BATCH):
        batch = texts[start : start + EMBEDDING_BATCH]
        chunk = embedder.encode(batch, show_progress_bar=False, convert_to_numpy=True)
        embeddings.append(chunk)
    return np.vstack(embeddings).astype(np.float32)

emb_corpus = corpus if EMBEDDING_MAX_DOCS is None else corpus.head(EMBEDDING_MAX_DOCS)
emb_doc_ids = emb_corpus["doc_id"].to_numpy()
emb_texts = ["passage: " + _row_text(row) for _, row in emb_corpus.iterrows()]

embeddings = embed_texts(emb_texts)
embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)

def e5_search_tool(query, top_k=10, agent_state=None):
    query_emb = embed_texts(["query: " + query])[0]
    query_emb = query_emb / (np.linalg.norm(query_emb) + 1e-12)
    scores = embeddings @ query_emb
    top_k = min(top_k, len(scores))
    if top_k == 0:
        return []
    top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
    top_sorted = top_idx[np.argsort(-scores[top_idx])]
    results = []
    for idx in top_sorted:
        row = emb_corpus.iloc[idx]
        results.append({
            "id": row.get("doc_id", idx),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": float(scores[idx]),
        })
    return results

e5_search_tool("salon chair", top_k=3)


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/Users/doug/ws/psuedorelevance-feedback/.venv/lib/python3.12/site-packages/searcharray/postings.py:506: UserWarning: Iterating over SearchArray is very slow and not reccomended.

            If you're looping a dataframe, slice out the non-SearchArray columns first.
  warnings.warn(warning_text)
/Users/doug/ws/psuedorelevance-feedback/.venv/lib/python3.12/site-packages/searcharray/postings.py:506: UserWarning: Iterating over SearchArray is very slow and not reccomended.

            If you're looping a dataframe, slice out the non-SearchArray columns first.
  warnings.warn(warning_text)


[{'id': np.int64(7465),
  'title': 'hair salon chair',
  'description': 'offers a wide selection of professional salon products including styling chair , salon hairdryer , salon equipment , etc . to showcase the uniqueness of each salon',
  'score': 0.8990092873573303},
 {'id': np.int64(33689),
  'title': 'office chair work beauty salon chair brown',
  'description': '',
  'score': 0.8987651467323303},
 {'id': np.int64(20026),
  'title': 'ginata salon beauty drafting chair',
  'description': '',
  'score': 0.8981103897094727}]

## Guard: avoid very similar queries (ELI5)

The config uses `disallow_similar_queries` with a 0.9 threshold. That means if the agent tries a nearly identical query, we ask it to be more creative.

ELI5: if the agent repeats itself, we politely say: try a different question.

In [6]:
import numpy as np
from cheat_at_search.embeddings import DEFAULT_MODEL_NAME, load_model

_guard_model = None

def _guard_embedder():
    global _guard_model
    if _guard_model is None:
        _guard_model = load_model(DEFAULT_MODEL_NAME)
    return _guard_model

def disallow_similar_queries_guard(query, tool_name, agent_state, threshold=0.9):
    if agent_state is None:
        return None
    past_queries = agent_state.setdefault("past_queries", {}).get(tool_name)
    if past_queries is None:
        past_queries = []
        agent_state["past_queries"][tool_name] = past_queries

    past_embeddings = agent_state.setdefault("past_query_embeddings", {}).get(tool_name)
    model = _guard_embedder()
    if past_embeddings is None and past_queries:
        past_embeddings = list(model.encode(past_queries))
        agent_state["past_query_embeddings"][tool_name] = past_embeddings

    if past_embeddings:
        query_embedding = np.asarray(model.encode(query))
        query_norm = float(np.linalg.norm(query_embedding))
        if query_norm > 0:
            for past_embedding in past_embeddings:
                past_embedding = np.asarray(past_embedding)
                past_norm = float(np.linalg.norm(past_embedding))
                if past_norm == 0:
                    continue
                similarity = float(np.dot(query_embedding, past_embedding) / (query_norm * past_norm))
                if similarity > threshold:
                    return "Error! You've already tried a very similar query. Be more creative and explore more!"

    query_embedding = np.asarray(model.encode(query))
    past_queries.append(query)
    agent_state["past_queries"][tool_name] = past_queries
    agent_state.setdefault("past_query_embeddings", {}).setdefault(tool_name, []).append(query_embedding)
    return None

def make_guarded_tool(tool_fn, tool_name, threshold=0.9):
    def guarded(query, top_k=10, agent_state=None):
        err = disallow_similar_queries_guard(query, tool_name, agent_state, threshold=threshold)
        if isinstance(err, str):
            return err
        return tool_fn(query, top_k=top_k, agent_state=agent_state)
    return guarded

bm25_guarded = make_guarded_tool(bm25_search_tool, "bm25", threshold=0.9)
e5_guarded = make_guarded_tool(e5_search_tool, "e5_base_v2", threshold=0.9)

bm25_guarded("salon chair", top_k=3, agent_state={})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'id': np.int64(7465),
  'title': 'hair salon chair',
  'description': 'offers a wide selection of professional salon products including styling chair , salon hairdryer , salon equipment , etc . to showcase the uniqueness of each salon',
  'score': 52.17097473144531},
 {'id': np.int64(22130),
  'title': 'height-adjustable stool salon chair',
  'description': 'here comes the stool , with 360-degree chair wheels , by which you can move freely with no noise in your space . with the comfortable table seat design , you will get a comfortable sitting experience . due to the stable and durable construction — the explosive gas rod passed germany tuv , and the high-strength steel frame — guarantees safer and longer use . top back to get one for your salon , dining table , bar table , etc .',
  'score': 44.94427490234375},
 {'id': np.int64(33689),
  'title': 'office chair work beauty salon chair brown',
  'description': '',
  'score': 39.476078033447266}]

## Stopper: require 6 tool calls (ELI5)

The config uses a stopper that only lets the loop end after 6 tool calls.

ELI5: we want the agent to try a few different searches before giving up.

In [7]:
def stop_after_tool_calls(num_tool_calls, min_calls=6):
    if num_tool_calls >= min_calls:
        return True
    return "You're doing really well. Please keep searching until 6 tool calls have been made so no stone is left unturned."

stop_after_tool_calls(3)

"You're doing really well. Please keep searching until 6 tool calls have been made so no stone is left unturned."

## Agentic strategy loop (ELI5)

We wrap the tools and the stopper into a loop. The agent keeps searching until it reaches the minimum number of tool calls.

In [8]:
from pydantic import BaseModel, Field
from cheat_at_search.agent.openai_agent import OpenAIAgent
from cheat_at_search.strategy import SearchStrategy

SYSTEM_PROMPT = """
You take user search queries and use a search tool to find products.

Look at the search tools you have, their limitations, how they work, etc when forming your plan.

Finally return results to the user per the SearchResults schema, ranked best to worst.

Gather results until you have 10 best matches you can find. It's important to return at least 10.

It's very important you consider carefully the correct ranking as you'll be evaluated on
how close that is to the average shoppers ideal ranking.
"""

class SearchResults(BaseModel):
    ranked_results: list[str] = Field(description="Top ranked search results (their doc_ids) when complete")

def _count_tool_calls(inputs):
    return sum(1 for item in inputs if isinstance(item, dict) and item.get("type") == "function_call_output")

class AgenticEcomStrategy(SearchStrategy):
    def __init__(self, corpus_df: pd.DataFrame, tools: list[callable], model: str = "gpt-5-mini", workers: int = 1):
        self.tools = tools
        self.model = model
        super().__init__(corpus_df, workers=workers)

    def search(self, query: str, k: int = 10):
        inputs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"The user's query: {query}"},
        ]
        agent_state = {}
        agent = OpenAIAgent(
            tools=self.tools,
            model=f"openai/{self.model}" if "/" not in self.model else self.model,
            response_model=SearchResults,
            reasoning_level="medium",
        )
        max_loops = 10
        for _ in range(max_loops):
            resp = agent.loop(inputs=inputs, agent_state=agent_state, logger=logger)
            tool_calls = _count_tool_calls(inputs)
            msg = stop_after_tool_calls(tool_calls, min_calls=6)
            if msg is True:
                break
            inputs.append({"role": "user", "content": msg})

        ranked = [doc_id for doc_id in (resp.ranked_results or [])]
        ranked = [doc_id_to_index.get(int(doc_id), -1) for doc_id in ranked if str(doc_id).isdigit()]
        ranked = [idx for idx in ranked if idx >= 0][:k]
        return ranked, [1.0] * len(ranked)

tools = [bm25_guarded, e5_guarded]
strategy = AgenticEcomStrategy(corpus, tools, workers=1)
strategy.search(queries.loc[0, "query"], k=5)

ValueError: Function guarded must have a docstring for tool description.

## Run the small benchmark

We run `run_strategy` on a small subset of queries and compute mean NDCG.

ELI5: NDCG is a score from 0 to 1 that says how good the ranking is (higher is better).

In [ ]:
from cheat_at_search.search import run_strategy, ndcgs

results = run_strategy(strategy, judgments, num_queries=QUERY_COUNT, seed=7, cache=False)
ndcg_series = ndcgs(results)
float(ndcg_series.mean())